# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

* **Unit of Analysis**: One row = one `content_hash_id` (URL) per `client_hash_id` on one `report_date`.
* **Table(s)**: `fact_daily` (primary performance logs).
* **Time Window**: Mid-panel month of March 2026 (`2026-03-01` to `2026-03-31`).
* **Label/Proxy**: `gsc_clicks` (Predicting search traffic volume).
* **Excluded**: `month` column (coarse leakage) and `ai_*` columns (to establish a baseline for organic search first).

In [15]:
import os, sys, subprocess, getpass
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/pretom26/ml_internship_flyrankAI.git"
REPO_DIR = Path("/content/ml_internship_flyrankAI")

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "skills/README.md").exists():
            os.chdir(candidate)
            break

%pip -q install duckdb

import duckdb
HF_TOKEN = None
if IN_COLAB:
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "fact_daily":  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}
MONTH_START = "2026-03-01"   # inclusive
MONTH_END   = "2026-04-01"   # exclusive mid-panel iteration month, never the sealed June sample
for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:12} {n:>12,} rows")


dim_clients           104 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily     78,835,655 rows


## 2. Fields: feature / label / context / excluded

* **Label**: `gsc_clicks` (Target for regression/ranking).
* **Features**: `gsc_impressions`, `gsc_avg_position`, `ga4_sessions`, `sessions_direct`, `client_has_ga4`.
* **Context**: `report_date`, `client_hash_id`, `content_hash_id`.
* **Excluded**: `ai_perplexity`, `ai_chatgpt` - Excluded because they represent a specific LLM-search slice that varies significantly by client adoption and requires separate modeling.

In [16]:
schema_probe = con.sql(f"SELECT * FROM {TABLES['fact_daily']} LIMIT 1").df()
print(list(schema_probe.columns))

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


### Grain Verification
We need to ensure that our 'Unit of Analysis' is truly unique. If a single URL (content_hash) appeared multiple times for the same client on the same day, our aggregations would be skewed. This query checks for any duplicates in our primary key combination.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [17]:
grain_check = con.sql(f"""
    WITH march AS (
        SELECT report_date, client_hash_id, content_hash_id
        FROM {TABLES['fact_daily']}
        WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MONTH_END}'
    )
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM march
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"duplicate (report_date, client, content) combinations found: {len(grain_check)}")
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate (report_date, client, content) combinations found: 0


,report_date,client_hash_id,content_hash_id,n


### 3.1 Data Availability Check
Before building features, we must know how much of our data actually contains the signals we need. Here, we verify how many rows in March 2026 have Google Search Console (GSC) data linked and ready for use.

### 3.2 The Leakage Experiment (The Trap)
In this step, we deliberately introduce a 'cheating' feature—using the target `gsc_clicks` to predict itself. By measuring the correlation, we see a perfect score of 1.0. This is a critical lesson: if a feature is too good to be true, it's likely 'leaking' information from the future that won't be available at the moment of prediction. We then remove it to keep the model honest.

In [18]:
# 3.1 Verification: Availability (filter with IS TRUE as requested)
avail_filtered = con.sql(f"""
    SELECT COUNT(*) as rows_surviving
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '{MONTH_START}' AND report_date < '{MONTH_END}'
    AND gsc_data_available IS TRUE
""").df()
display(avail_filtered)

# 3.2 Feature Frame with Explanations
# 1. gsc_impressions: Knowable because it is logged by Google at the end of the search day.
# 2. gsc_avg_position: Knowable because it represents the ranking state for that specific day.
# 3. ga4_sessions: Knowable because analytics sessions are counted in real-time as users arrive.
# 4. sessions_direct: Knowable because it captures baseline non-search intent from existing users.
# 5. client_has_ga4: Knowable because it is a static technical configuration property of the client.

honest_features = con.sql(f"""
    SELECT
        gsc_impressions, gsc_avg_position, ga4_sessions, sessions_direct, client_has_ga4::INT as has_ga4
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '{MONTH_START}' AND report_date < '{MONTH_END}'
    AND gsc_data_available IS TRUE
    LIMIT 5
""").df()
display(honest_features)

# 3.3 The Trap: Leakage Experiment
df_leak = con.sql(f"""
    SELECT
        gsc_clicks AS gsc_clicks_LEAK, -- THE TRAP
        gsc_clicks AS target           -- Label
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '{MONTH_START}' AND report_date < '{MONTH_END}'
    LIMIT 10000
""").df()

leakage_corr = df_leak['gsc_clicks_LEAK'].corr(df_leak['target'])
print(f"Correlation with leakage column: {leakage_corr:.4f} (Score jump to perfect!)")

# Cleanup: Leakage lesson performed, trap deleted.
del df_leak

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_surviving
0,3611061


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_impressions,gsc_avg_position,ga4_sessions,sessions_direct,has_ga4
0,20,3.350000,<NA>,<NA>,0
1,1,0.000000,<NA>,<NA>,0
2,125,4.928000,<NA>,<NA>,0
3,7,4.000000,<NA>,<NA>,0
4,11,2.272727,<NA>,<NA>,0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Correlation with leakage column: 1.0000 (Score jump to perfect!)


### Slice Summary
Finally, we summarize our working dataset for March. This confirms the total row count, the number of unique content pieces, and the active client count to ensure our 'lane' of the data river is flowing as expected.

## 4. Data limits

**Limitation**: Sparse historical GSC data. Many content items may exist in the warehouse via GA4 sessions but lack `gsc_data_available = TRUE` for older dates, leading to a sample bias toward newer or high-performing pages.

In [19]:
slice_summary = con.sql(f"""
    SELECT
        COUNT(*)                          AS n_rows,
        COUNT(DISTINCT content_hash_id)   AS n_content_items,
        COUNT(DISTINCT client_hash_id)    AS n_clients,
        MIN(report_date)                  AS min_date,
        MAX(report_date)                  AS max_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MONTH_END}'
      AND gsc_impressions > 0
""").df()

slice_summary


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_content_items,n_clients,min_date,max_date
0,3611061,176738,47,2026-03-01,2026-03-31


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.